# Lab 1.2 — Feature Engineering from Infrastructure Telemetry

**Module 1 | Day 1 | AI/ML Intermediate Workshop — Nutanix Engineering**

---

## What Is Feature Engineering?

Feature engineering is the process of using domain knowledge to transform raw data into representations that make machine learning algorithms work better. In practice, **feature engineering has more impact on model performance than algorithm choice**.

Andrew Ng famously said: *'Coming up with features is difficult, time-consuming, requires expert knowledge. Applied machine learning is basically feature engineering.'*

In the infrastructure world, this means taking raw telemetry — CPU percentages, log levels, response times — and turning them into signals that reveal patterns like:
- Is this node nearing a failure boundary?
- Is this anomaly a recurring pattern or a one-off spike?
- Which component is most correlated with degraded performance?

## Learning Objectives

By the end of this lab you will be able to:

1. Extract and engineer **time-based features** from infrastructure timestamps
2. Create **aggregated / rolling window features** that capture temporal behavior per host
3. Encode **categorical variables** (log levels, component names, hostnames) correctly
4. Apply and compare **three scalers** and understand when each is appropriate
5. Use **log transforms** and **derived features** to fix skewed distributions
6. Apply **PCA** for dimensionality reduction and visualization
7. Preview **feature importance** using a quick Random Forest model

---

> **Instructor Note:** Lab 1.1 covered data cleaning. Remind participants that clean data is a prerequisite — garbage in, garbage out. This lab assumes the cleaned CSV exists, but will generate synthetic data if it does not, so every participant can follow along regardless of Lab 1.1 completion status.

**Estimated time:** 90 minutes

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| pandas | `pandas` |
| numpy | `numpy` |
| matplotlib | `matplotlib` |
| seaborn | `seaborn` |
| scikit-learn | `scikit-learn` |

**Install all at once:**
```bash
pip install pandas numpy matplotlib seaborn scikit-learn
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


## Setup: Import Libraries

We import everything we need upfront. Key libraries:
- **pandas / numpy**: data manipulation
- **matplotlib / seaborn**: visualization
- **sklearn.preprocessing**: scalers and encoders
- **sklearn.decomposition**: PCA
- **sklearn.ensemble**: RandomForest for feature importance preview

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    LabelEncoder
)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Consistent plot styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All libraries loaded successfully.')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

---
## Step 1: Load or Generate Cleaned Infrastructure Data

We first try to load the output from **Lab 1.1** (`cleaned_server_logs.csv`). If that file is not available, we generate statistically equivalent synthetic data so this lab runs standalone.

The dataset represents telemetry from a 10-node Nutanix cluster — each row is one log event with associated host metrics captured at that moment.

> **Instructor Note:** Point out the `try/except` pattern — production pipelines always need fallback strategies. This is also a good moment to discuss data lineage: the output of one lab should be the input of the next.

In [ ]:
import os

# ── Try to load Lab 1.1 output ───────────────────────────────────────────────
DATA_PATH = 'cleaned_server_logs.csv'

try:
    df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
    print(f'Loaded {DATA_PATH} from Lab 1.1  →  shape: {df.shape}')
    DATA_SOURCE = 'Lab 1.1 CSV'

except FileNotFoundError:
    print(f'{DATA_PATH} not found — generating synthetic Nutanix telemetry inline.\n')
    DATA_SOURCE = 'Synthetic (standalone mode)'

    # ── Synthetic data parameters ─────────────────────────────────────────────
    N_ROWS = 2_000

    HOSTS      = [f'ntnx-cvm-{i:03d}' for i in range(1, 11)]   # 10 CVMs
    COMPONENTS = ['AOS', 'Prism', 'Stargate', 'Cerebro',
                  'Curator', 'Acropolis']                         # Nutanix stack
    LOG_LEVELS  = ['DEBUG', 'INFO', 'WARNING', 'ERROR']
    LOG_WEIGHTS = [0.35, 0.40, 0.15, 0.10]                       # realistic dist

    # Timestamps: 30 days of data at irregular intervals
    start = pd.Timestamp('2024-01-01')
    timestamps = pd.to_datetime(
        np.sort(np.random.uniform(0, 30 * 24 * 3600, N_ROWS)),
        unit='s', origin=start
    )

    hosts      = np.random.choice(HOSTS, N_ROWS)
    log_levels = np.random.choice(LOG_LEVELS, N_ROWS, p=LOG_WEIGHTS)
    components = np.random.choice(COMPONENTS, N_ROWS)

    # CPU: higher when ERROR, with some natural variation
    cpu_base    = np.where(log_levels == 'ERROR', 75, 45)
    cpu_percent = np.clip(cpu_base + np.random.normal(0, 15, N_ROWS), 5, 100)

    # Memory: correlated with CPU but more stable
    memory_mb = np.clip(
        8192 + cpu_percent * 80 + np.random.normal(0, 1500, N_ROWS),
        2048, 65536
    )

    # Disk I/O: spiky log-normal distribution
    disk_io_mbps = np.clip(
        np.random.lognormal(mean=3.5, sigma=0.8, size=N_ROWS), 0.5, 800
    )

    # Response time: right-skewed; worse during ERROR events
    rt_base          = np.where(log_levels == 'ERROR', 800, 150)
    response_time_ms = np.clip(
        rt_base + np.abs(np.random.exponential(scale=200, size=N_ROWS)),
        10, 10_000
    )

    # Error codes: only populated for ERROR rows
    error_codes = np.where(
        log_levels == 'ERROR',
        np.random.choice(['E001', 'E002', 'E003', 'E004', 'E005'], N_ROWS),
        ''
    )

    df = pd.DataFrame({
        'timestamp'       : timestamps,
        'host'            : hosts,
        'log_level'       : log_levels,
        'component'       : components,
        'cpu_percent'     : cpu_percent.round(2),
        'memory_mb'       : memory_mb.round(0).astype(int),
        'disk_io_mbps'    : disk_io_mbps.round(2),
        'response_time_ms': response_time_ms.round(1),
        'error_code'      : error_codes,
    })

    # Pre-compute calendar columns Lab 1.1 would have added
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Mon … 6=Sun

    print(f'Synthetic dataset created  →  shape: {df.shape}')

print(f'\nData source : {DATA_SOURCE}')
print(f'Shape       : {df.shape}')
print(f'\nColumn dtypes:')
print(df.dtypes)

In [ ]:
# Quick sanity check — see the first few rows
df.head(8)

In [ ]:
# Statistical summary of numeric columns
df[['cpu_percent', 'memory_mb', 'disk_io_mbps', 'response_time_ms']].describe().round(2)

---
## Step 2: Time-Based Feature Engineering

Infrastructure anomalies are highly temporal. CPU spikes on Friday evenings before a weekend, storage rebalancing jobs run at 2 AM, and business-hours traffic can be 3x higher than off-hours. Encoding *when* something happened is as important as encoding *what* happened.

We engineer:

| Feature | Description |
|---|---|
| `hour_of_day` | 0-23 — captures intra-day patterns |
| `day_of_week` | 0 (Mon) - 6 (Sun) |
| `is_weekend` | Binary flag: Sat or Sun |
| `is_business_hours` | Binary flag: 09:00-17:00 weekdays |
| `time_since_last_error_s` | Seconds since previous ERROR on the same host |

> **Instructor Note:** `time_since_last_error_s` is a **lag feature** — it lets the model know whether a node is in a recovery window after a recent error. This kind of feature is impossible to derive from a single row in isolation; it requires groupby + shift logic. This is where domain knowledge is irreplaceable.

In [ ]:
# ── Ensure timestamp column is datetime ──────────────────────────────────────
df['timestamp'] = pd.to_datetime(df['timestamp'])

# ── Calendar features ────────────────────────────────────────────────────────
df['hour_of_day']       = df['timestamp'].dt.hour
df['day_of_week']       = df['timestamp'].dt.dayofweek    # 0=Mon … 6=Sun

# Weekend flag: Saturday=5, Sunday=6
df['is_weekend']        = (df['day_of_week'] >= 5).astype(int)

# Business hours: 09:00-17:00 on weekdays
df['is_business_hours'] = (
    (df['hour_of_day'] >= 9) &
    (df['hour_of_day'] <  17) &
    (df['day_of_week'] <  5)
).astype(int)

print('Calendar features added.')
print(df[['timestamp', 'hour_of_day', 'day_of_week',
          'is_weekend', 'is_business_hours']].head(6))

In [ ]:
# ── Lag feature: time since last ERROR per host ──────────────────────────────
# Sort by host + timestamp so the shift is chronological
df = df.sort_values(['host', 'timestamp']).reset_index(drop=True)

# Mark ERROR rows only; others become NaT
df['error_timestamp'] = df['timestamp'].where(df['log_level'] == 'ERROR')

# Forward-fill within each host group so every row knows the last error time
df['last_error_time'] = (
    df.groupby('host')['error_timestamp']
      .transform(lambda x: x.ffill())
)

# Compute seconds since last error
# NaN means no prior error observed for this host in the dataset
df['time_since_last_error_s'] = (
    (df['timestamp'] - df['last_error_time'])
    .dt.total_seconds()
)

# Sentinel value -1: 'no previous error recorded for this host'
df['time_since_last_error_s'] = (
    df['time_since_last_error_s'].fillna(-1).round(1)
)

# Drop helper columns
df.drop(columns=['error_timestamp', 'last_error_time'], inplace=True)

print('Lag feature (time_since_last_error_s) added.')
print(df[['host', 'timestamp', 'log_level',
          'time_since_last_error_s']].head(12))

In [ ]:
# ── Visualise: ERROR log distribution by hour of day and day of week ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hour of day distribution
error_by_hour = (
    df[df['log_level'] == 'ERROR']
    .groupby('hour_of_day')
    .size()
    .reindex(range(24), fill_value=0)
)
axes[0].bar(error_by_hour.index, error_by_hour.values,
            color='salmon', edgecolor='white')
axes[0].axvspan(9, 17, alpha=0.15, color='green', label='Business hours')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('ERROR Count')
axes[0].set_title('ERROR Events by Hour of Day')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()

# Day of week distribution
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
error_by_dow = (
    df[df['log_level'] == 'ERROR']
    .groupby('day_of_week')
    .size()
    .reindex(range(7), fill_value=0)
)
bar_colors = ['#4C72B0'] * 5 + ['#DD8452', '#DD8452']   # weekends highlighted
axes[1].bar(range(7), error_by_dow.values,
            color=bar_colors, edgecolor='white')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('ERROR Count')
axes[1].set_title('ERROR Events by Day of Week')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)

plt.suptitle('Temporal Distribution of ERROR Logs — Time Features Matter!',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f'Total rows with time features: {df.shape[0]} | Columns now: {df.shape[1]}')

---
## Step 3: Aggregated / Rolling Window Features

A single log event is a snapshot. ML models often need **context**: what was the average CPU over the last 5 events on this host? How many errors has this component generated in total? Aggregated features encode behavioral trends that single-row values cannot.

We create:

| Feature | Logic |
|---|---|
| `cpu_rolling_mean_5` | Rolling 5-row mean of cpu_percent per host |
| `memory_rolling_mean_5` | Rolling 5-row mean of memory_mb per host |
| `error_rate_per_host` | Total ERROR count for this host in the dataset |
| `component_error_count` | Total ERROR count for this component |

> **Instructor Note:** We use a row-count window (`window=5`) as a proxy for a time window. In production Nutanix telemetry pipelines (e.g., Pulse or Xi Insights), you would use a proper time-based window like `pd.Grouper(freq='10min')`. The concept is identical — we are just simplifying for lab purposes. Ask the class: what happens to this feature when one host emits 10x more log events than another?

In [ ]:
# ── Rolling window features (per host, sorted chronologically) ───────────────
# Data is already sorted by host + timestamp from Step 2

# Rolling 5-row mean for cpu_percent within each host
df['cpu_rolling_mean_5'] = (
    df.groupby('host')['cpu_percent']
      .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
      .round(2)
)

# Rolling 5-row mean for memory_mb within each host
df['memory_rolling_mean_5'] = (
    df.groupby('host')['memory_mb']
      .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
      .round(0)
)

print('Rolling window features added.')
print('Side-by-side comparison for one host:')
sample_host = df['host'].iloc[0]
df[df['host'] == sample_host][
    ['timestamp', 'host', 'cpu_percent', 'cpu_rolling_mean_5',
     'memory_mb', 'memory_rolling_mean_5']
].head(10)

In [ ]:
# ── Error rate per host ──────────────────────────────────────────────────────
host_error_counts = (
    df[df['log_level'] == 'ERROR']
    .groupby('host')
    .size()
    .rename('error_rate_per_host')
)
df = df.merge(host_error_counts, on='host', how='left')
df['error_rate_per_host'] = df['error_rate_per_host'].fillna(0).astype(int)

# ── Error count per component ─────────────────────────────────────────────────
component_error_counts = (
    df[df['log_level'] == 'ERROR']
    .groupby('component')
    .size()
    .rename('component_error_count')
)
df = df.merge(component_error_counts, on='component', how='left')
df['component_error_count'] = df['component_error_count'].fillna(0).astype(int)

print('Aggregation features added.')
print('\nErrors per host:')
print(df.groupby('host')['error_rate_per_host'].first().sort_values(ascending=False))
print('\nErrors per component:')
print(df.groupby('component')['component_error_count'].first().sort_values(ascending=False))

In [ ]:
# ── Visualise: Error rates across the cluster ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-host error rates
host_summary = (
    df.groupby('host')['error_rate_per_host']
      .first()
      .sort_values(ascending=True)
)
axes[0].barh(host_summary.index, host_summary.values,
             color='steelblue', edgecolor='white')
axes[0].set_xlabel('Total ERROR Events')
axes[0].set_title('ERROR Count per Host (CVM Node)')
axes[0].axvline(host_summary.mean(), color='red', linestyle='--',
                label=f'Mean = {host_summary.mean():.0f}')
axes[0].legend()

# Per-component error rates
comp_summary = (
    df.groupby('component')['component_error_count']
      .first()
      .sort_values(ascending=True)
)
axes[1].barh(comp_summary.index, comp_summary.values,
             color='coral', edgecolor='white')
axes[1].set_xlabel('Total ERROR Events')
axes[1].set_title('ERROR Count per Nutanix Component')
axes[1].axvline(comp_summary.mean(), color='navy', linestyle='--',
                label=f'Mean = {comp_summary.mean():.0f}')
axes[1].legend()

plt.suptitle('Aggregated Error Features — Contextual Signals for the Model',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Shape after Step 3: {df.shape}')

---
## Step 4: Encoding Categorical Variables

Machine learning algorithms operate on numbers. Categorical variables — like `log_level`, `component`, and `host` — must be converted to numeric form. The wrong encoding choice can introduce false ordinal relationships or explode dimensionality.

| Strategy | When to use | Example |
|---|---|---|
| **One-Hot Encoding** | Nominal categories with no inherent order; small unique value count | `log_level`, `component` |
| **Label Encoding** | High-cardinality nominal, or tree-based model input | `host` (10 nodes) |
| **Ordinal Encoding** | Categories with a meaningful order | `log_level` treated as severity scale |

> **Instructor Note:** Ask the class: *What is the danger of Label Encoding `log_level` as 0=DEBUG, 1=INFO, 2=WARNING, 3=ERROR?*
> Answer: a linear model would treat ERROR as exactly 3x DEBUG in severity, and assume equal spacing — which is completely arbitrary. One-hot encoding avoids this artifact entirely. Tree models do not care because they split on thresholds, not magnitudes.

In [ ]:
# ── Keep original shape for before/after comparison ─────────────────────────
shape_before = df.shape

# ── One-Hot Encode: log_level and component ──────────────────────────────────
# drop_first=False: keep all levels for interpretability in a teaching context.
# In production you would typically use drop_first=True to reduce collinearity.
df = pd.get_dummies(
    df,
    columns=['log_level', 'component'],
    prefix=['loglevel', 'comp'],
    dtype=int     # 0/1 integers, not booleans
)

ohe_cols = [c for c in df.columns
            if c.startswith('loglevel_') or c.startswith('comp_')]

print('One-hot encoding applied to log_level and component.')
print(f'New OHE columns: {ohe_cols}')
print(f'\nShape before OHE : {shape_before}')
print(f'Shape after OHE  : {df.shape}')

In [ ]:
# ── Label Encode: host ───────────────────────────────────────────────────────
# LabelEncoder assigns an integer to each unique host string.
# Appropriate here because: (a) tree models are the likely consumers,
# (b) adding 10 OHE columns for host would inflate dimensionality for small gain.

le = LabelEncoder()
df['host_encoded'] = le.fit_transform(df['host'])

# Print the mapping for full transparency
host_map = dict(zip(le.classes_, le.transform(le.classes_)))
print('Label encoding: host  →  host_encoded')
print('Mapping:', host_map)
print(f'\ndf.shape after label encoding: {df.shape}')

In [ ]:
# Preview: host encoding + OHE columns side by side
encoded_cols = ['host', 'host_encoded'] + ohe_cols
df[encoded_cols].head(6)

---
## Step 5: Feature Scaling

Many ML algorithms are sensitive to the **scale** of input features. A `memory_mb` column with values in the thousands will dominate a `cpu_percent` column in the single digits when computing distances or gradients — unless we normalize.

| Algorithm type | Scaling needed? |
|---|---|
| KNN, SVM, PCA, K-Means, Neural Networks | **Yes** — distance / gradient sensitive |
| Decision Trees, Random Forest, XGBoost | **No** — split-based, scale invariant |
| Linear / Logistic Regression, Lasso | **Yes** — coefficient magnitudes affected |

We compare three scalers:

| Scaler | Formula | Best for |
|---|---|---|
| **StandardScaler** | `(x - mean) / std` | Normally distributed features |
| **MinMaxScaler** | `(x - min) / (max - min)` | Known bounded range; no outliers |
| **RobustScaler** | `(x - median) / IQR` | **Infrastructure telemetry** — outlier-prone |

> **Instructor Note:** Infrastructure metrics are notoriously spiky. A single CPU runaway event at 99.9% can make StandardScaler compress 95% of the rest of your data into a tiny range near the mean. RobustScaler uses the median and interquartile range, making it resistant to those spikes. This should be your default for telemetry data unless you have strong reasons otherwise.

In [ ]:
# ── Columns to scale ────────────────────────────────────────────────────────
scale_cols = ['cpu_percent', 'memory_mb', 'disk_io_mbps', 'response_time_ms']

# ── Instantiate three scalers ────────────────────────────────────────────────
std_scaler    = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

# Fit and transform — store results in separate DataFrames for comparison
df_std    = pd.DataFrame(
    std_scaler.fit_transform(df[scale_cols]),
    columns=[f'{c}_std'    for c in scale_cols]
)
df_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(df[scale_cols]),
    columns=[f'{c}_mm'     for c in scale_cols]
)
df_robust = pd.DataFrame(
    robust_scaler.fit_transform(df[scale_cols]),
    columns=[f'{c}_robust' for c in scale_cols]
)

print('All three scalers fitted.')
print('\nOriginal — descriptive statistics:')
print(df[scale_cols].describe().round(2))

In [ ]:
# ── Visual comparison: before vs. after each scaler ─────────────────────────
# Focus on cpu_percent and response_time_ms as the most illustrative examples

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

scalers = [
    ('StandardScaler',  df_std,    'cpu_percent_std',    'response_time_ms_std'),
    ('MinMaxScaler',    df_minmax, 'cpu_percent_mm',     'response_time_ms_mm'),
    ('RobustScaler',    df_robust, 'cpu_percent_robust', 'response_time_ms_robust'),
]

orig_cpu = df['cpu_percent']
orig_rt  = df['response_time_ms']

for row_idx, (name, scaled_df, cpu_col, rt_col) in enumerate(scalers):
    ax_cpu = axes[row_idx, 0]
    ax_rt  = axes[row_idx, 1]

    ax_cpu.hist(orig_cpu, bins=40, alpha=0.4, color='grey',
                density=True, label='Original')
    ax_cpu.hist(scaled_df[cpu_col], bins=40, alpha=0.7,
                color='steelblue', density=True, label=name)
    ax_cpu.set_title(f'cpu_percent — {name}')
    ax_cpu.set_xlabel('Value')
    ax_cpu.set_ylabel('Density')
    ax_cpu.legend(fontsize=9)

    ax_rt.hist(orig_rt, bins=40, alpha=0.4, color='grey',
               density=True, label='Original')
    ax_rt.hist(scaled_df[rt_col], bins=40, alpha=0.7,
               color='coral', density=True, label=name)
    ax_rt.set_title(f'response_time_ms — {name}')
    ax_rt.set_xlabel('Value')
    ax_rt.set_ylabel('Density')
    ax_rt.legend(fontsize=9)

plt.suptitle('Scaler Comparison: Original vs. Scaled Distributions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Append RobustScaler columns to the main dataframe ────────────────────────
# We choose RobustScaler as our canonical scaled features for downstream use
df = pd.concat([df.reset_index(drop=True),
                df_robust.reset_index(drop=True)], axis=1)

robust_cols = [c for c in df.columns if c.endswith('_robust')]
print('RobustScaler features appended.')
print('New scaled columns:', robust_cols)
print(f'df.shape: {df.shape}')

---
## Step 6: Feature Transformation and Derived Features

Many infrastructure metrics are **right-skewed** — most values are small but occasional spikes create a long tail. This violates the normality assumptions of several algorithms and concentrates the distribution in ways that make learning difficult.

We also create **derived features** — combinations of existing columns that encode domain knowledge directly into the feature space. This is where infrastructure expertise becomes a competitive ML advantage.

| Derived Feature | Formula | Intuition |
|---|---|---|
| `log_response_time` | `log(response_time_ms + 1)` | Fix right-skew; latencies often follow a log-normal distribution |
| `cpu_memory_ratio` | `cpu_percent / (memory_mb / 1024)` | CPU per GB — reveals memory-starved-but-low-CPU nodes |
| `io_per_cpu` | `disk_io_mbps / (cpu_percent + 1)` | I/O intensity relative to CPU activity |
| `is_high_cpu` | `cpu_percent > 80` | Binary flag for threshold-based anomalies |
| `is_slow_response` | `response_time_ms > 500` | Flag for SLA-breach events |

> **Instructor Note:** The `+1` in `log(x+1)` is called log1p — it handles zero values gracefully. NumPy provides `np.log1p()` for this. Always check for zeros or negatives before applying a log transform. The threshold values (80% CPU, 500ms) are typical Nutanix operational guidelines — in a real deployment you would derive these from SLA contracts.

In [ ]:
# ── Log transform: response_time_ms ─────────────────────────────────────────
# np.log1p = log(1 + x), safe when x >= 0 (avoids log(0) = -inf)
df['log_response_time'] = np.log1p(df['response_time_ms'])

# ── Derived ratio features ───────────────────────────────────────────────────
# CPU load per GB of RAM — high value means CPU is working hard relative to available memory
df['cpu_memory_ratio'] = (
    df['cpu_percent'] / (df['memory_mb'] / 1024.0 + 1e-6)  # 1e-6 avoids div/0
).round(4)

# Disk I/O intensity relative to CPU activity
df['io_per_cpu'] = (
    df['disk_io_mbps'] / (df['cpu_percent'] + 1.0)  # +1 when cpu_percent can be 0
).round(4)

# ── Binary threshold flags ───────────────────────────────────────────────────
# Encoding operational thresholds as model-ready signals
df['is_high_cpu']      = (df['cpu_percent']      > 80).astype(int)
df['is_slow_response'] = (df['response_time_ms'] > 500).astype(int)

new_features = ['log_response_time', 'cpu_memory_ratio', 'io_per_cpu',
                'is_high_cpu', 'is_slow_response']
print('Derived features added:')
print(df[new_features].describe().round(3))

In [ ]:
# ── Visualise: before vs. after log transform on response_time_ms ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before: right-skewed raw distribution
axes[0].hist(df['response_time_ms'], bins=60,
             color='salmon', edgecolor='white')
axes[0].set_title('response_time_ms — Original (Right-Skewed)')
axes[0].set_xlabel('Response Time (ms)')
axes[0].set_ylabel('Count')
skew_before = df['response_time_ms'].skew()
axes[0].text(0.65, 0.85, f'Skewness: {skew_before:.2f}',
             transform=axes[0].transAxes, fontsize=11,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

# After: log1p transform brings it toward normal
axes[1].hist(df['log_response_time'], bins=60,
             color='mediumseagreen', edgecolor='white')
axes[1].set_title('log_response_time — After log1p Transform')
axes[1].set_xlabel('log(1 + Response Time)')
axes[1].set_ylabel('Count')
skew_after = df['log_response_time'].skew()
axes[1].text(0.05, 0.85, f'Skewness: {skew_after:.2f}',
             transform=axes[1].transAxes, fontsize=11,
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.suptitle('Log Transform Fixes Right-Skewed Response Time Distribution',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Skewness reduced from {skew_before:.2f}  →  {skew_after:.2f}')

In [ ]:
# ── Visualise: binary flag distributions ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, color in zip(axes,
                           ['is_high_cpu', 'is_slow_response'],
                           ['steelblue', 'coral']):
    counts = df[col].value_counts().sort_index()
    labels = ['Normal', 'Flagged']
    ax.bar(labels, counts.values, color=[color, 'crimson'], edgecolor='white')
    ax.set_title(f'{col} Distribution')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, f'{v:,}\n({v/len(df)*100:.1f}%)',
                ha='center', fontsize=10)

plt.suptitle('Binary Threshold Features — Encoding Domain Knowledge as ML Signals',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 7: Dimensionality Reduction with PCA

After all our feature engineering, the dataset now has many columns. **Principal Component Analysis (PCA)** finds the directions of maximum variance in the data and projects it onto a lower-dimensional space.

**Two uses of PCA:**
1. **Visualization** — project to 2D to see cluster structure (what we do here)
2. **Actual dimensionality reduction** — feed PCA components into a model to reduce noise and collinearity

**Intuition:** Imagine each feature as an axis in N-dimensional space. PCA rotates the axes to align with the directions where the data spreads the most. The first principal component captures the most variance, the second captures the next most, and so on.

> **Instructor Note:** Before applying PCA, features **must** be scaled. PCA is variance-based, so unscaled features with large magnitudes (like `memory_mb` in the thousands) would dominate all principal components and make the decomposition meaningless. Also emphasize the interpretability trade-off: **PCA components are not interpretable as original features** — that is the cost of the compression.

In [ ]:
# ── Select numeric features for PCA ─────────────────────────────────────────
pca_feature_cols = [
    'cpu_percent', 'memory_mb', 'disk_io_mbps', 'response_time_ms',
    'hour_of_day', 'day_of_week', 'is_weekend', 'is_business_hours',
    'time_since_last_error_s', 'cpu_rolling_mean_5', 'memory_rolling_mean_5',
    'error_rate_per_host', 'component_error_count',
    'log_response_time', 'cpu_memory_ratio', 'io_per_cpu',
    'is_high_cpu', 'is_slow_response',
]

# Drop rows with NaN in these columns (should be very few)
pca_data = df[pca_feature_cols].dropna()
print(f'Rows available for PCA: {len(pca_data)} of {len(df)}')

# Scale before PCA — critical step
pca_scaler = StandardScaler()
X_scaled   = pca_scaler.fit_transform(pca_data)

# Fit full PCA first to inspect explained variance across all components
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

# Reduce to 2 components for 2D visualization
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca  = pca_2d.fit_transform(X_scaled)

print(f'PCA fitted on {len(pca_data)} rows x {len(pca_feature_cols)} features.')
print(f'2D PCA explains {pca_2d.explained_variance_ratio_.sum()*100:.1f}% of total variance.')
print(f'PC1: {pca_2d.explained_variance_ratio_[0]*100:.1f}%  |  PC2: {pca_2d.explained_variance_ratio_[1]*100:.1f}%')

In [ ]:
# ── Explained variance ratio plot (Scree plot + Cumulative) ─────────────────
n_show   = min(15, len(pca_feature_cols))
evr      = pca_full.explained_variance_ratio_[:n_show]
cum_evr  = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot: variance per component
axes[0].bar(range(1, n_show + 1), evr * 100,
            color='steelblue', edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot — Variance per Component')
axes[0].set_xticks(range(1, n_show + 1))

# Cumulative explained variance
axes[1].plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
             cum_evr * 100, 'o-', color='coral', linewidth=2, markersize=5)
axes[1].axhline(90, color='grey',  linestyle='--', label='90% threshold')
axes[1].axhline(95, color='navy',  linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].set_ylim([0, 102])

# Annotate how many components reach 90% / 95%
n_90 = int(np.argmax(cum_evr >= 0.90)) + 1
n_95 = int(np.argmax(cum_evr >= 0.95)) + 1
axes[1].axvline(n_90, color='grey', linestyle=':', alpha=0.8)
axes[1].axvline(n_95, color='navy', linestyle=':', alpha=0.8)

plt.suptitle('PCA Explained Variance — How Many Components Do We Need?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Components needed for 90% variance: {n_90}')
print(f'Components needed for 95% variance: {n_95}')

In [ ]:
# ── 2D PCA scatter plot colored by log_level ─────────────────────────────────
# Recover original log_level from OHE columns if present
loglevel_ohe_cols = [c for c in df.columns if c.startswith('loglevel_')]

if loglevel_ohe_cols:
    # Reconstruct label from argmax of OHE block
    ll_recon = (
        df.loc[pca_data.index, loglevel_ohe_cols]
          .idxmax(axis=1)
          .str.replace('loglevel_', '', regex=False)
    )
else:
    ll_recon = df.loc[pca_data.index, 'log_level']

pca_plot_df = pd.DataFrame({
    'PC1'      : X_pca[:, 0],
    'PC2'      : X_pca[:, 1],
    'log_level': ll_recon.values
})

color_map = {
    'DEBUG'  : '#4C72B0',
    'INFO'   : '#55A868',
    'WARNING': '#C44E52',
    'ERROR'  : '#DD8452',
}

fig, ax = plt.subplots(figsize=(10, 7))
for level, color in color_map.items():
    mask = pca_plot_df['log_level'] == level
    if mask.any():
        ax.scatter(
            pca_plot_df.loc[mask, 'PC1'],
            pca_plot_df.loc[mask, 'PC2'],
            c=color, label=level, alpha=0.55, s=20, edgecolors='none'
        )

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title(
    'PCA 2D Projection — Colored by Log Level\n'
    'Visible separation between ERROR and DEBUG/INFO suggests learnable structure',
    fontsize=12
)
ax.legend(title='Log Level', markerscale=2)
plt.tight_layout()
plt.show()
print('If ERROR points cluster separately from INFO/DEBUG, the engineered features have discriminative power.')

---
## Step 8: Feature Importance Preview

Before wrapping up, let us answer the most practical question: **which features actually matter for predicting anomalies?**

We train a quick `RandomForestClassifier` to predict whether a log event is an ERROR (binary classification), then plot the feature importances. This validates that our engineered features carry real predictive signal.

> **Instructor Note:** This is a **teaser** for Module 2. Random Forest importance is based on mean decrease in impurity (Gini impurity). It can be biased toward high-cardinality continuous features. In Module 2 we cover **SHAP values** — a game-theory-based method that provides more reliable, per-prediction explanations. Think of this bar chart as the first look before the deep dive. Also: `error_rate_per_host` being high importance is a data leakage signal — we computed it over the full dataset, including the test set. In production, compute aggregations only on past data.

In [ ]:
# ── Build binary target: is_error ───────────────────────────────────────────
if 'loglevel_ERROR' in df.columns:
    y = df['loglevel_ERROR'].values
elif 'log_level' in df.columns:
    y = (df['log_level'] == 'ERROR').astype(int).values
else:
    raise ValueError('Cannot find log_level or loglevel_ERROR column.')

# ── Feature matrix ───────────────────────────────────────────────────────────
fi_features = [
    'cpu_percent', 'memory_mb', 'disk_io_mbps', 'response_time_ms',
    'hour_of_day', 'day_of_week', 'is_weekend', 'is_business_hours',
    'time_since_last_error_s', 'cpu_rolling_mean_5', 'memory_rolling_mean_5',
    'error_rate_per_host', 'component_error_count',
    'log_response_time', 'cpu_memory_ratio', 'io_per_cpu',
    'is_high_cpu', 'is_slow_response', 'host_encoded',
]

X_fi = df[fi_features].fillna(0).values

# Stratified split preserves class balance in both train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_fi, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Shallow forest: fast, avoids overfitting on this lab dataset
# class_weight='balanced' handles imbalance — ERRORs are ~10% of rows
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train)
test_acc  = rf.score(X_test,  y_test)
print(f'Random Forest trained.')
print(f'Train accuracy: {train_acc:.3f}  |  Test accuracy: {test_acc:.3f}')
print('Note: accuracy alone is misleading on imbalanced data — Module 2 covers precision/recall/F1.')

In [ ]:
# ── Feature importance bar chart ─────────────────────────────────────────────
importances        = pd.Series(rf.feature_importances_, index=fi_features)
importances_sorted = importances.sort_values(ascending=True)

# Color engineered features differently from raw features
engineered_set = {
    'time_since_last_error_s', 'cpu_rolling_mean_5', 'memory_rolling_mean_5',
    'error_rate_per_host', 'component_error_count', 'log_response_time',
    'cpu_memory_ratio', 'io_per_cpu', 'is_high_cpu', 'is_slow_response',
}
colors = ['#DD8452' if feat in engineered_set else '#4C72B0'
          for feat in importances_sorted.index]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(importances_sorted.index, importances_sorted.values,
               color=colors, edgecolor='white', height=0.7)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4C72B0', label='Raw feature'),
    Patch(facecolor='#DD8452', label='Engineered feature (created in this lab)'),
]
ax.legend(handles=legend_elements, loc='lower right')

ax.set_xlabel('Mean Decrease in Impurity (Feature Importance)')
ax.set_title(
    'Random Forest Feature Importance for ERROR Prediction\n'
    'Orange = features we engineered today',
    fontsize=12, fontweight='bold'
)
ax.set_xlim(0, importances_sorted.values.max() * 1.18)

for bar, val in zip(bars, importances_sorted.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8.5)

plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(importances.sort_values(ascending=False).head(5).round(4).to_string())
print('\nComing in Module 2: SHAP values for per-prediction explanations.')

---
## Step 9: Save the Feature-Engineered Dataset

We save the final feature-enriched dataframe as `features_engineered.csv`. This file becomes the input for all Module 2 labs.

> **Instructor Note:** In a production MLOps pipeline, this save step would write to a **feature store** (e.g., Feast, Tecton, or a cloud blob store like Nutanix Objects). The feature store ensures that training and serving pipelines use identical transformations — preventing **train/serve skew**, one of the most common and painful production ML bugs. The CSV approach here is pedagogically convenient but would not scale to petabyte telemetry.

In [ ]:
# ── Drop columns not needed for model training ───────────────────────────────
cols_to_drop = []

# Raw timestamp: not useful directly as a model feature after extraction
if 'timestamp' in df.columns:
    cols_to_drop.append('timestamp')

# String host: replaced by host_encoded
if 'host' in df.columns:
    cols_to_drop.append('host')

# Raw error_code: sparse string, needs separate encoding — out of scope
if 'error_code' in df.columns:
    cols_to_drop.append('error_code')

df_final = df.drop(columns=cols_to_drop, errors='ignore').copy()

# Save to CSV
OUTPUT_PATH = 'features_engineered.csv'
df_final.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'Final shape: {df_final.shape}')

In [ ]:
# ── Lab pipeline summary ─────────────────────────────────────────────────────
print('=' * 68)
print('  LAB 1.2 SUMMARY — Feature Engineering Pipeline')
print('=' * 68)

steps = [
    ('Original data (input)',
     'timestamp, host, log_level, component, cpu_percent, memory_mb,',
     'disk_io_mbps, response_time_ms, error_code  (9 columns)'),
    ('Step 2: Time features',
     'hour_of_day, day_of_week, is_weekend,',
     'is_business_hours, time_since_last_error_s  (+5)'),
    ('Step 3: Rolling / aggregation',
     'cpu_rolling_mean_5, memory_rolling_mean_5,',
     'error_rate_per_host, component_error_count  (+4)'),
    ('Step 4: Categorical encoding',
     'loglevel_DEBUG/INFO/WARNING/ERROR  (+4)',
     'comp_AOS/Prism/Stargate/Cerebro/Curator/Acropolis  (+6), host_encoded  (+1)'),
    ('Step 5: RobustScaler',
     'cpu_percent_robust, memory_mb_robust,',
     'disk_io_mbps_robust, response_time_ms_robust  (+4)'),
    ('Step 6: Transforms + derived',
     'log_response_time, cpu_memory_ratio, io_per_cpu,',
     'is_high_cpu, is_slow_response  (+5)'),
]

for step, line1, line2 in steps:
    print(f'  {step}')
    print(f'    {line1}')
    print(f'    {line2}')
    print()

print(f'  Output file    : {OUTPUT_PATH}')
print(f'  Final shape    : {df_final.shape[0]:,} rows x {df_final.shape[1]} columns')
print(f'  Features added : {df_final.shape[1] - 9} (from original 9)')
print('=' * 68)

---
## Checkpoint: Discussion Questions

Take 8-10 minutes to discuss these questions with your table before the instructor debrief.

---

**Q1: Rolling window gotcha**

In Step 3 we used a row-count window (`window=5`) rather than a time-based window. What problem does this create in production? In a Nutanix cluster where one CVM emits 100 log events per minute and another emits 1 per hour, does `window=5` represent the same time interval for both hosts?

---

**Q2: Label encoding and linear models**

We label-encoded `host` as integers 0-9. If we then trained a **linear regression** using these encoded hosts as a feature, what assumption would the model incorrectly learn? What alternative encoding strategy would you use instead, and what is the trade-off in terms of column count?

---

**Q3: Scaler choice under outliers**

Your team's SRE tells you that during a storage migration, `disk_io_mbps` hit 980 MB/s for 20 minutes — roughly 10x the normal range. You are about to train a KNN-based anomaly detector. Which scaler should you use, and why? What happens to the 980 MB/s outlier under each of the three scalers we tested?

---

**Q4: Feature importance and data leakage**

In Step 8, `error_rate_per_host` likely ranks very high in feature importance. But does this mean *hosts with more errors cause more errors*? Why is this circular reasoning, and what should you do about it in a production anomaly detector? *(Hint: think about when the aggregation was computed relative to the train/test split.)*

---
## Key Takeaways

### What We Built
Starting from cleaned Nutanix CVM telemetry (9 raw columns), we constructed a rich feature matrix ready for ML model training. Every transformation was grounded in infrastructure domain knowledge.

### Core Principles

| Principle | Lab Evidence |
|---|---|
| **Domain knowledge beats algorithm choice** | `time_since_last_error_s` captures recovery windows — no algorithm discovers this from raw data alone |
| **Encode temporal context explicitly** | ERRORs cluster at predictable hours; `is_business_hours` gives the model this signal directly |
| **Match your scaler to your data** | `RobustScaler` survives infrastructure spikes; `MinMaxScaler` collapses under a single outlier event |
| **Fix skewed distributions before training** | `log1p(response_time_ms)` brought skewness from >3.0 toward ~0.5; raw exponential tails confuse gradient-based learners |
| **Validate engineered features** | Random Forest importance confirmed that engineered features (orange bars) carry significant predictive signal |
| **Watch for data leakage** | `error_rate_per_host` was computed on the whole dataset — in production, compute only on past-window data to avoid look-ahead bias |

### What Is Next

- **Module 2, Lab 2.1** — Anomaly Detection: using these features with Isolation Forest and DBSCAN to flag unhealthy CVM nodes
- **Module 2, Lab 2.2** — SHAP Values: explaining *why* the model flagged a specific node, going beyond aggregate feature importance
- **Module 3** — Time-Series Forecasting: using proper time-indexed windows to predict resource exhaustion before it happens

---

> **Instructor Note:** Collect the `features_engineered.csv` output path from participants before moving to Module 2. A consistent, clean feature set is required for the anomaly detection labs. Participants who ran in standalone (synthetic) mode will have numerically different values but structurally identical columns.

---
*Lab 1.2 complete — Output: `features_engineered.csv`*

---
## 🎯 Your Turn — Challenges

These challenges use the **`features_engineered` dataframe** and scalers created in this lab.  
No starter code — just the problem. Refer back to the steps above if you need a reminder.

### Challenge 1 — Memory Utilisation Bins

**Task:**  
Create a new column `memory_utilisation_pct` = `memory_mb / 32768 * 100` (assuming 32 GB max RAM per CVM).

Then use `pd.cut()` to bin it into three categories:
- `Low`: 0–30%
- `Medium`: 30–70%  
- `High`: >70%

Answer: What percentage of CVMs fall into each bin? Which host has the most `High` utilisation events?

*Hint: `pd.cut(bins=[0,30,70,100], labels=['Low','Medium','High'])`*

In [ ]:
# Challenge 1 — Your solution here




### Challenge 2 — Scaler Comparison on an Outlier Host

In Step 5 we compared three scalers. Now apply this knowledge:

**Task:**  
Pick the host with the **highest max cpu_percent** in the dataset.  
For that host's rows only, apply `StandardScaler` and `RobustScaler` separately to `cpu_percent`.

Plot both scaled distributions side by side (histogram).  
Answer: Which scaler handles that host's outlier CPU spikes better, and why?

*Hint: `df[df['host'] == chosen_host]`, fit scaler on that subset, `.fit_transform()`*

In [ ]:
# Challenge 2 — Your solution here




### Challenge 3 — Composite Anomaly Flag

**Task:**  
Create a composite boolean column `is_critical` that is `True` when ALL three conditions are met:
1. `is_high_cpu` == 1  
2. `is_slow_response` == 1  
3. `log_level_ERROR` == 1 (from one-hot encoded column)

Then:
1. What percentage of all rows are `is_critical`?
2. Which `component` has the highest `is_critical` rate?
3. Add `is_critical` as a feature and re-run the Random Forest from Step 8. Does feature importance change?

*This is the kind of domain-driven feature an SRE would define — and it often outperforms purely statistical features.*

In [ ]:
# Challenge 3 — Your solution here


